# Use case 1 (research): rebuild my own first paper, from prompts only

**You drive the agent. It writes the code.** This notebook holds the prompts and nothing else. Paste each one into Claude Code with this notebook open, let it fill in the empty cell below, then apply the check before moving on.

In 2015, as a second-year PhD student in this department, my first paper was [ARGO](https://www.pnas.org/doi/10.1073/pnas.1515373112) (Yang, Santillana, Kou, *PNAS* 2015): use Google search volume, plus the disease's own history, to nowcast influenza. It took the better part of a year. We are going to rebuild it in four prompts, and **download every byte of the data ourselves**.

> Stuck, or your agent is not cooperating? The worked version with every output is in [`01_research_dengue_soln.ipynb`](01_research_dengue_soln.ipynb). You will not be blocked.

## Prompt 0: get the data, from scratch

Two sources, nothing pre-supplied.

```
Use the pytrends package to download monthly Google search interest in Mexico for
"dengue", "sintomas de dengue", "mosquito" and "dengue sintomas", over the longest
window it will give me in one request. Save it to data/gt_dengue_mx.csv.
```

```
Download the OpenDengue national extract (V1.3) from
https://github.com/OpenDengue/master-repo, pull out Mexico's weekly records, and
aggregate them to monthly totals. Drop any month with fewer than 4 weeks of data.
Save to data/opendengue_mexico_monthly.csv.
```

**Expect the first one to fail.** Let the agent read the traceback and fix itself; that loop is most of why a terminal agent beats an autocomplete pane. Then ask what it gave up to make the error go away.

In [ ]:
# Prompt 0: paste the agent's download code here and run it.


## The prompt that matters most, and it comes first

Before any modeling, interrogate what you just downloaded.

```
Check the coverage of what you just downloaded. Are the months contiguous? Is the
case definition the same throughout? Compare the overlapping period against
data/MX_Dengue_trends.csv, the file I already had. Do the two agree?
```

**Your check, not the agent's.** Three things are worth finding here, and none of them announce themselves:

* Is the record actually continuous, or is it islands with years missing in between?
* Does the *meaning* of the number change partway through?
* When two official sources cover the same month, do they agree? Aggregate agreement can hide month-level disagreement, and monthly is the resolution you are about to model at.

In [ ]:
# Check the coverage and the agreement. Do not skip this.


Whatever you found, pick **one contiguous stretch with one consistent case definition** and model only that. Write down why you chose it.

In [ ]:
# Restrict to your chosen window, then plot cases against search interest.


## Prompt 1: the honest baseline

```
Fit ordinary least squares of cases on the single "dengue" search column, training
only on the first 24 months. Predict the rest from that frozen fit. Do not refit on
anything later.
```

That last sentence is load-bearing. Leakage is the most common way this goes quietly wrong, and an agent optimizing for a good-looking number will refit if you let it.

In [ ]:
# Prompt 1: paste the agent's code here and run it.


## Prompt 2: now make it ARGO

```
Now build the ARGO model from Yang, Santillana and Kou (PNAS 2015). Work in log
space. Regress log cases on the logs of all four search terms plus three
autoregressive lags of log cases. Use L1 regularization with a cross-validated
penalty, and retrain on a rolling 24-month window at every step so the model only
ever sees the past. Fit two references on the identical rolling scheme:
autoregression only, and search only. Return predictions on the original scale.
```

Two structural ideas carry ARGO and both are in that paragraph: **autoregression** and **dynamic training**.

In [ ]:
# Prompt 2: paste the agent's code here and run it.


## Check it again: what did it decide for you?

```
Walk me through what you just did, line by line. Where did you have to make a choice
I did not specify? What would break if my data were slightly different?
```

Then verify the answer instead of taking it. If it says anything about the search columns, go and count:

In [ ]:
# How many search values are exactly zero, and what share of the record is that?


Hold two things at once. The agent's patch was probably **correct**. It was also probably **silent**, and a silent correct patch and a silent wrong one look identical from the outside.

> The agent closes the gap between having an idea and seeing a number, almost completely. It does not close the gap between seeing a number and believing it. That gap is still the job.

## Prompt 3: score everything against each other

```
Build one comparison table over the common evaluation window: RMSE, MAE and
correlation for the static baseline, the autoregression-only model, the search-only
model, and ARGO. Add a column giving each model's RMSE relative to the
autoregression benchmark. Then plot ARGO against the truth over time.
```

In [ ]:
# Prompt 3: paste the agent's code here and run it.


## Now read the table honestly

The agent will not do this part, and it is the part that decides whether the analysis is any good. Write down your own answers before looking at mine:

* Which single change bought the most accuracy?
* How much did search add **over and above** the disease's own history?
* How does search do **on its own**? Is it sufficient, or only complementary?
* How many evaluation points is this margin based on? Is that enough to claim anything?

**Then compare with [`01_research_dengue_soln.ipynb`](01_research_dengue_soln.ipynb)**, which has the executed outputs and my reading of them.

---

**Next:** [`02_education_reading_group.ipynb`](02_education_reading_group.ipynb), where the agent is not writing models at all.